# 02 - 数据预处理

点云预处理：地面滤除 → 离群点去除 → 树木分割 → 参数提取

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import laspy
from pathlib import Path

from src.data.preprocessing import (
    PointCloudPreprocessor,
    remove_ground,
    remove_outliers,
    cluster_trees,
    compute_tree_params,
)

In [ ]:
# 加载数据
las_file = Path('../data/00_raw').glob('*.las')
las = laspy.read(str(next(las_file)))
raw_points = np.vstack([las.x, las.y, las.z]).T
print(f'原始点数: {len(raw_points):,}')

In [ ]:
# 预处理pipeline
preprocessor = PointCloudPreprocessor(
    voxel_size=0.02,
    cluster_eps=0.6,
    cluster_min=100,
)

results = preprocessor.process(raw_points)
print(f'检测到 {len(results)} 棵树')

In [ ]:
# 查看每棵树的参数
import pandas as pd

df = pd.DataFrame([{
    'tree_id': r['tree_id'],
    'height': r['params']['height'],
    'dbh': r['params']['dbh_estimate'],
    'crown_width': r['params']['crown_width'],
    'n_points': r['params']['n_points'],
} for r in results])

print(df.to_string(index=False))